In [1]:
import scanpy as sc
import harmonypy as hm
import tqdm as notebook_tqdm
import phate
import pandas as pd
from matplotlib.colors import to_rgba
from pygam import LinearGAM, s
import meld
import scipy.stats as stats
import numpy as np
import sklearn as sk
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
from itertools import combinations
import gseapy as gp
from scipy.stats import ttest_ind
from statannotations.Annotator import Annotator
import scprep

In [2]:
adata= sc.read_h5ad("/Users/Jan/data/final_final_data.h5ad")

In [ ]:
# Figure 3A
baseline= "1-WT-No-Tx"
meld_scores_list= []
conditions= ['1-WT-1h', '1-WT-4h', '1-KA-No-Tx', '1-KA-1h', '1-KA-4h']
for condition in conditions:
    adata.obs["meld"] = np.nan        
    adata_filtered_by_condition= adata[adata.obs["Sample name"].isin([condition, baseline])]
    meld_op = meld.MELD(random_state= 12)
    sample_densities= meld_op.fit_transform(adata_filtered_by_condition, sample_labels=adata_filtered_by_condition.obs['Sample name'])
    sample_densities= sk.preprocessing.normalize(
        sample_densities,
        norm='l1',
        axis=1
    )
    densities_df= pd.DataFrame(
        sample_densities,
        index= adata_filtered_by_condition.obs_names,
        columns= sorted([baseline, condition])
    )
    meld_score = pd.DataFrame({
        "MELD Likelihood": densities_df[condition],
        #"Comparison": f"{condition} vs {baseline}"
        "Comparison": condition
    })
    meld_scores_list.append(meld_score)
meld_scores= pd.concat(meld_scores_list)
comparison_order = [cond for cond in conditions if cond!= baseline]
comparison_map = {c: str(i) for i, c in enumerate(comparison_order)}
meld_scores["comparison_sort"] = meld_scores["Comparison"].map(comparison_map)
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].astype('category')
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].cat.reorder_categories([str(i) for i in range(len(comparison_order))])
fig, ax = plt.subplots(figsize=(10, 10))
ax= scprep.plot.jitter(meld_scores["comparison_sort"], meld_scores["MELD Likelihood"], 
                    ax= ax,
                   plot_means=False)
ax.set_ylim(0, 1)
ax.set_xticklabels(pd.unique(meld_scores.sort_values("comparison_sort")["Comparison"]))
plt.xlabel("Comparison vs WT No-Tx")
plt.title("MELD Likelihoods of Different Conditions vs WT Untreated")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3D part 1
meld_scores["Polarisation"]= (meld_scores["MELD Likelihood"]- 0.5).abs()
summary_stats= meld_scores.groupby("Comparison")["Polarisation"].mean().reset_index()
print(summary_stats)

In [ ]:
# Figure 3B
adata_wt= adata[adata.obs['Sample Type']== 'WT']
timepoints= adata_wt.obs['Sample name'].unique()
pairs= [(timepoints[0], timepoints[2]), (timepoints[1], timepoints[0]), (timepoints[1], timepoints[2])]
meld_scores_list= []
for group_a, group_b in pairs:
    pair_mask= adata_wt.obs['Sample name'].isin([group_a, group_b])
    subset_pair= adata_wt[pair_mask].copy()
    meld_op = meld.MELD(random_state= 12)
    sample_densities= meld_op.fit_transform(subset_pair, sample_labels=subset_pair.obs['Sample name'])
    sample_densities= sk.preprocessing.normalize(
        sample_densities,
        norm='l1',
        axis=1
    )
    cols= sorted([group_a, group_b])
    densities_df = pd.DataFrame(sample_densities, index=subset_pair.obs_names, columns=cols)
    col_name= f"meld_{group_a}_vs_{group_b}"
    adata_wt.obs[col_name]= np.nan
    adata_wt.obs.loc[subset_pair.obs_names, col_name]= densities_df[group_b]
    meld_score = pd.DataFrame({
        "MELD Likelihood": densities_df[group_b],
        "Comparison": f"{group_a} vs {group_b}"
    })
    meld_scores_list.append(meld_score)
meld_scores= pd.concat(meld_scores_list)
comparison_order = [f"{a} vs {b}" for a, b in pairs]
comparison_map = {c: str(i) for i, c in enumerate(comparison_order)}
meld_scores["comparison_sort"] = meld_scores["Comparison"].map(comparison_map)
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].astype('category')
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].cat.reorder_categories([str(i) for i in range(len(comparison_order))])
fig, ax = plt.subplots(figsize=(10, 10))
scprep.plot.jitter(meld_scores["comparison_sort"], meld_scores["MELD Likelihood"], 
                    ax= ax,
                   plot_means=False)
ax.set_ylim(0, 1)
ax.set_xticklabels(pd.unique(meld_scores.sort_values("comparison_sort")["Comparison"]))#, rotation=90)
plt.xlabel("Comparison")
plt.title("MELD Likelihoods of Different Pairwise Comparisons between WT conditions")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3D part 2
meld_scores["Polarisation"]= (meld_scores["MELD Likelihood"]- 0.5).abs()
summary_stats= meld_scores.groupby("Comparison")["Polarisation"].mean().reset_index()
print(summary_stats)

In [ ]:
# Figure 3C
adata_ka= adata[adata.obs['Sample Type']== 'KA']
vmin, vmax= 0, 1
timepoints= adata.obs['Sample name'].unique()
pairs= [(timepoints[0], timepoints[2]), (timepoints[1], timepoints[0]), (timepoints[1], timepoints[2])]
meld_scores_list= []
for group_a, group_b in pairs:
    pair_mask= adata.obs['Sample name'].isin([group_a, group_b])
    subset_pair= adata[pair_mask].copy()
    meld_op = meld.MELD(random_state= 12)
    sample_densities= meld_op.fit_transform(subset_pair, sample_labels=subset_pair.obs['Sample name'])
    sample_densities= sk.preprocessing.normalize(
        sample_densities,
        norm='l1',
        axis=1
    )
    cols= sorted([group_a, group_b])
    densities_df = pd.DataFrame(sample_densities, index=subset_pair.obs_names, columns=cols)
    col_name= f"meld_{group_a}_vs_{group_b}"
    adata.obs[col_name]= np.nan
    adata.obs.loc[subset_pair.obs_names, col_name]= densities_df[group_b]
    meld_score = pd.DataFrame({
        "MELD Likelihood": densities_df[group_b],
        "Comparison": f"{group_a} vs {group_b}"
    })
    meld_scores_list.append(meld_score)
meld_scores= pd.concat(meld_scores_list)
comparison_order = [f"{a} vs {b}" for a, b in pairs]
comparison_map = {c: str(i) for i, c in enumerate(comparison_order)}
meld_scores["comparison_sort"] = meld_scores["Comparison"].map(comparison_map)
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].astype('category')
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].cat.reorder_categories([str(i) for i in range(len(comparison_order))])
fig, ax = plt.subplots(figsize=(10, 10))
scprep.plot.jitter(meld_scores["comparison_sort"], meld_scores["MELD Likelihood"], 
                    ax= ax,
                   plot_means=False)
ax.set_ylim(0, 1)
ax.set_xticklabels(pd.unique(meld_scores.sort_values("comparison_sort")["Comparison"]))
plt.xlabel("Comparison")
plt.title("MELD Likelihoods of Different Pairwise Comparisons between KA conditions")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3D part 3
meld_scores["Polarisation"]= (meld_scores["MELD Likelihood"]- 0.5).abs()
summary_stats= meld_scores.groupby("Comparison")["Polarisation"].mean().reset_index()
print(summary_stats)

In [ ]:
# Figures 3E- G
vmin, vmax= 0, 1
timepoints = sorted(adata_wt.obs['Sample name'].unique())
pairs= [(timepoints[0], timepoints[2]), (timepoints[1], timepoints[0]), (timepoints[1], timepoints[2])]
for group_a, group_b in pairs:
    pair_mask= adata_wt.obs['Sample name'].isin([group_a, group_b])
    subset_pair= adata_wt[pair_mask].copy()
    try:
        meld_op = meld.MELD()
        sample_densities= meld_op.fit_transform(subset_pair, sample_labels=subset_pair.obs['Sample name'])
        sample_densities= sk.preprocessing.normalize(
            sample_densities,
            norm='l1',
            axis=1
        )
        cols= sorted([group_a, group_b])
        densities_df = pd.DataFrame(sample_densities, index=subset_pair.obs_names, columns=cols)
        col_name= f"meld_{group_a}_vs_{group_b}"
        adata_wt.obs[col_name]= np.nan
        adata_wt.obs.loc[subset_pair.obs_names, col_name]= densities_df[group_b]
        sc.pl.embedding(
            adata_wt, 
            basis='phate', 
            color=col_name,
            cmap='magma',
            title=f"{group_a} vs {group_b} MELD density",
            frameon=False,
            na_color= '#e0e0e0',
            vmin= vmin,
            vmax= vmax,
            show= True,
        )
    except Exception as e:
        print(f"Error: {e}")